# Cross-Model Solver Test

Tests `CrossModelSolver` on a set of preset target words to see how often it lands on the correct answer.
Models are loaded once and the solver state is reset between runs. Each target is tested multiple times
(since random guesses introduce variance) to get a reliable success rate.

In [ ]:
import io
import contextlib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import gensim.downloader as api
from sentence_transformers import SentenceTransformer
from semantle.semantle import Semantle
from solver import CrossModelSolver

# Load once — this takes a while (word2vec + sentence-transformer + encoding)
game = Semantle()
solver = CrossModelSolver(game, steepness=10.0)

In [ ]:
#Load both googlenews model and all-MiniLM-L6-v2 models

model2_name = "all-MiniLM-L6-v2"
model1 = api.load("word2vec-google-news-300")
model2 = SentenceTransformer(model2_name)

In [ ]:
# Load common words
word_list = pd.read_csv("common_word_list.csv")
word_list = word_list["words"].tolist()

# Model 2: get embeddings for all words
embeddings = model2.encode(word_list)

# For every pair of words, get sim score for Model 2
data = []
for i in range(len(word_list)):
    for j in range(i+1, len(word_list)):
        similarity_score_all_MiniLM_L6_v2 = model2.similarity(embeddings[i], embeddings[j]).item()
        data.append([f"{word_list[i]}_{word_list[j]}", similarity_score_all_MiniLM_L6_v2])

df = pd.DataFrame(data, columns=["pair", "model2_sim_score"])

# Model 1: get similarity score for all pairs
df["googlenews_sim_score"] = df["pair"].apply(lambda x: model1.similarity(x.split("_")[0], x.split("_")[1]).item())

# Sort
df = df.sort_values(by="googlenews_sim_score", ascending=False)

In [ ]:
# Plot similarity scores
sns.scatterplot(x="googlenews_sim_score", y="model2_sim_score", data=df, alpha=0.2)
plt.xlim(-.2, 1)
plt.ylim(-.2, 1)
plt.xlabel("Google News Sim Score")
plt.ylabel("Model 2 Sim Score")
plt.title("Similarity Scores for Model 1 and Model 2")
plt.show()


In [ ]:
df.head(20)

In [ ]:
TARGET_WORDS = [
    "medical",
    "computer",
    "river",
    "music",
    "kitchen",
    "science",
    "happy",
    "animal",
    "weather",
    "sport",
    "reflect",
    "admit",
    "cold",
    "building",
    "plant",
    "window",
    "night",
    "bring",
    "camera",
    "cultural"]

In [ ]:
RUNS_PER_WORD = 5

def run_on_target(solver, target):
    """Reset solver state for a new target and run, returning (answer, correct, n_guesses, log)."""
    solver.semantle.word_of_the_day = target
    solver.guesses = []
    solver.log_probs = np.zeros(len(solver.vocab))
    solver.target_idx = solver.word_to_idx.get(target)

    log = io.StringIO()
    with contextlib.redirect_stdout(log):
        answer = solver.solve(max_rounds=1000)

    n_guesses = len(solver.guesses)
    correct = (answer == target)
    return answer, correct, n_guesses, log.getvalue()


results = {}  # target -> list of (answer, correct, n_guesses)
for target in TARGET_WORDS:
    runs = []
    for run in range(RUNS_PER_WORD):
        answer, correct, n_guesses, log = run_on_target(solver, target)
        runs.append((answer, correct, n_guesses, log))
    results[target] = runs

    wins = sum(c for _, c, _, _ in runs)
    avg_guesses = sum(g for _, _, g, _ in runs) / len(runs)
    answers = [a for a, _, _, _ in runs]
    print(f"  {target:12s}  {wins}/{RUNS_PER_WORD} correct  avg_guesses={avg_guesses:.0f}  answers={answers}")

    # For correct runs, show first 6 and last 6 guesses
    for i, (a, c, g, log) in enumerate(runs):
        if not c:
            continue
        round_lines = [l for l in log.splitlines() if l.strip().startswith("Round")]
        first = round_lines[:6]
        last = round_lines[-6:]
        print(f"    --- run {i+1} ({g} guesses) ---")
        for line in first:
            print(f"    {line.strip()}")
        if len(round_lines) > 12:
            print(f"    ... ({len(round_lines) - 12} rounds omitted) ...")
        for line in last:
            if line not in first:
                print(f"    {line.strip()}")

In [ ]:
total_runs = sum(len(runs) for runs in results.values())
total_correct = sum(sum(c for _, c, _, _ in runs) for runs in results.values())
total_guesses = sum(sum(g for _, _, g, _ in runs) for runs in results.values())

print(f"Overall: {total_correct}/{total_runs} correct ({100*total_correct/total_runs:.0f}%)")
print(f"Average guesses: {total_guesses/total_runs:.0f}")

In [ ]:
# Get a measure of the variation of the number of guesses
variation = []
for target in TARGET_WORDS:
    runs = results[target]
    guesses = [g for _, _, g, _ in runs]
    variation.append(np.std(guesses))
print(f"Variation of guesses: {np.mean(variation)}")

